# Lesson 26: Fine-tuning LLMs cheaply — PEFT, LoRA and QLoRA 🪡
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain why **full fine-tuning** is expensive and what **PEFT** (Parameter-Efficient Fine-Tuning) does instead
2. Understand **LoRA**'s low-rank trick with a hands-on calculation
3. Fine-tune a real LLM with LoRA to give it a new "personality"
4. Load a model in **4-bit** for **QLoRA**, and compare memory use

**⏱️ Timing:** why PEFT 10' | LoRA maths 10' | LoRA fine-tune 30' | QLoRA 10'

> ⚙️ **GPU required:** Runtime → Change runtime type → **T4 GPU**.

## 🧥 An analogy: altering a suit
- **Full fine-tuning** = sewing a whole new suit. You must update **every** weight → needs huge memory, and you save a whole new copy of the model.
- **PEFT** = taking the suit to a tailor for small **alterations**. **Freeze** the original model and train a few small extra pieces.
- **LoRA** (Low-Rank Adaptation, 2021) = the most popular alteration: add two **thin** matrices next to each big weight matrix.
- **QLoRA** (2023) = LoRA + storing the frozen model in **4-bit** numbers → fine-tune billion-parameter models on a single gaming-sized GPU.

| | Full fine-tuning | LoRA | QLoRA |
|---|---|---|---|
| Weights trained | 100% | often < 1% | often < 1% |
| Frozen model stored in | 16/32-bit | 16-bit | **4-bit** |
| File you save | whole model (GBs) | small adapter (MBs) | small adapter (MBs) |

## Part 1: The LoRA trick in numbers 🧮
A weight matrix **W** might be 4096 × 4096. Instead of changing W directly, LoRA learns **W + B·A**, where **B** is 4096 × r and **A** is r × 4096, with a small **rank r** (like 8).

In [ ]:
import numpy as np
d, r = 4096, 8
full = d * d
lora = d * r + r * d
print(f"Full update: {full:,} numbers | LoRA (r={r}): {lora:,} numbers → {lora / full:.2%} of the size")

# Low rank = a big grid built from a few simple patterns
B = np.random.randn(6, 1); A = np.random.randn(1, 6)
print("\nA 6×6 matrix made from just 6 + 6 numbers (rank 1):\n", (B @ A).round(1))

**Why does this work?** Research found that the *change* needed to adapt a model to a new task is usually "simple" — it can be described by a few directions (low rank), even though the model itself is huge.

### 🧩 Exercise 1
Try `r = 1, 4, 16, 64`. How does the share of trainable numbers grow? What might you lose with a very small r?

## Part 2: LoRA fine-tune a real LLM 🔧
**Goal:** teach `Qwen2.5-0.5B-Instruct` a new personality — **"Captain Kiwi"**, a pirate who answers science questions in pirate talk and always ends with *"Arrr!"* We'll train on a tiny handmade dataset.

In [ ]:
!pip -q install peft bitsandbytes
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=torch.float32, device_map="auto")

def chat(m, question, max_new_tokens=80):
    text = tok.apply_chat_template([{"role": "user", "content": question}], tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("BEFORE:", chat(model, "Why is the sky blue?"))

In [ ]:
data = [
    ("Why is the sky blue?", "Ahoy! Sunlight be bouncin' off tiny bits of air, and blue light scatters the most across the sky, matey. Arrr!"),
    ("What is gravity?", "Gravity be the invisible rope that pulls everythin' toward everythin' else — that's why ye fall off the plank! Arrr!"),
    ("How do plants make food?", "Plants be cookin' with sunlight, water and air in their green leaves — photosynthesis, they call it. Arrr!"),
    ("What is an atom?", "An atom be the tiniest bit o' treasure that matter is built from, with a heart o' protons and neutrons. Arrr!"),
    ("Why does ice float?", "Ice be lighter than the same amount o' water, so it bobs on top like a cork on the sea. Arrr!"),
    ("What causes thunder?", "Lightning heats the air so fast it bursts outward with a mighty BOOM across the sky. Arrr!"),
    ("How do magnets work?", "Magnets have two ends, north and south — opposites pull together and the same ends push apart, like rival crews. Arrr!"),
    ("What is the Moon?", "The Moon be Earth's rocky shipmate, sailin' round us every month and pullin' on the tides. Arrr!"),
    ("Why do we sweat?", "Sweat be yer body's sea spray — when it dries off yer skin, it carries the heat away. Arrr!"),
    ("What is DNA?", "DNA be the secret treasure map inside yer cells that tells yer body how to build ye. Arrr!"),
    ("How fast does light travel?", "Light sails at about 300,000 kilometres every second — faster than any ship on the seven seas! Arrr!"),
    ("What is a volcano?", "A volcano be a mountain with a fiery belly o' melted rock that sometimes bursts out the top. Arrr!"),
]

def make_example(q, a):
    msgs = [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    full = tok.apply_chat_template(msgs, tokenize=False)
    prompt_len = len(tok(tok.apply_chat_template(msgs[:1], tokenize=False, add_generation_prompt=True))["input_ids"])
    ids = tok(full, return_tensors="pt")["input_ids"][0]
    labels = ids.clone(); labels[:prompt_len] = -100        # only learn the ANSWER, not the question
    return ids, labels

examples = [make_example(q, a) for q, a in data]
print(len(examples), "training examples")

In [ ]:
config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
                    target_modules=["q_proj", "v_proj"],     # add LoRA next to the attention Query and Value matrices
                    task_type="CAUSAL_LM")
lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

In [ ]:
import random
optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=1e-3)
lora_model.train()
for epoch in range(8):
    random.shuffle(examples); total = 0
    for ids, labels in examples:
        out = lora_model(input_ids=ids[None].to(lora_model.device), labels=labels[None].to(lora_model.device))
        optimizer.zero_grad(); out.loss.backward(); optimizer.step()
        total += out.loss.item()
    print(f"epoch {epoch + 1}: average loss {total / len(examples):.3f}")
lora_model.eval();

In [ ]:
for q in ["Why is the sky blue?", "What is a black hole?", "How do vaccines work?"]:     # the last two were NOT in training!
    print(f"❓ {q}\n🏴‍☠️ {chat(lora_model, q)}\n")

In [ ]:
# Switch the adapter off → the original model is still there, untouched
with lora_model.disable_adapter():
    print("ADAPTER OFF:", chat(lora_model, "What is a black hole?"))

lora_model.save_pretrained("captain_kiwi_adapter")
!du -sh captain_kiwi_adapter
print("(The full model is about 2 GB in 32-bit. The adapter is tiny!)")

### 🧩 Exercise 2
1. Did Captain Kiwi generalise to questions **not** in the training data?
2. Check the science in its answers. Did fine-tuning on style make it **less accurate** anywhere?
3. Change `r=8` to `r=2` or `r=32`, or add `"k_proj", "o_proj"` to `target_modules`. What changes?
4. Make your own personality dataset (a sports commentator? a Shakespearean actor?) with 12 examples and train it.

## Part 3: QLoRA — the 4-bit trick 🗜️
QLoRA stores the frozen model in **4-bit** numbers (16 possible values per weight instead of millions), but trains the LoRA adapter in higher precision.

In [ ]:
from transformers import BitsAndBytesConfig
import gc

def gpu_memory_mb(m):
    return m.get_memory_footprint() / 1e6

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(NAME, quantization_config=bnb, device_map="auto")
print(f"32-bit model: {gpu_memory_mb(model):7.0f} MB")
print(f" 4-bit model: {gpu_memory_mb(model_4bit):7.0f} MB")
print("4-bit answer:", chat(model_4bit, "Why is the sky blue?", 50))

In [ ]:
from peft import prepare_model_for_kbit_training
model_4bit = prepare_model_for_kbit_training(model_4bit)
qlora = get_peft_model(model_4bit, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
qlora.print_trainable_parameters()
opt = torch.optim.AdamW([p for p in qlora.parameters() if p.requires_grad], lr=1e-3)
qlora.train()
for epoch in range(8):
    for ids, labels in examples:
        out = qlora(input_ids=ids[None].to(qlora.device), labels=labels[None].to(qlora.device))
        opt.zero_grad(); out.loss.backward(); opt.step()
qlora.eval()
print("QLoRA Captain Kiwi:", chat(qlora, "What is a black hole?"))

### 🚀 Challenge
Use the memory table from Lesson 24: what's the biggest model you could **QLoRA fine-tune** on a free T4? Try `Qwen/Qwen2.5-1.5B-Instruct` with QLoRA.

### ⚖️ When should you fine-tune at all?
| Want the model to… | Best tool |
|---|---|
| Follow a format or style, or play a persona | **Fine-tuning** (LoRA) or a good system prompt |
| Know **new or private facts** (your school's timetable) | **RAG** (Lesson 32) — fine-tuning is unreliable for facts |
| Do something once | Just **prompt** it (Lessons 28–29) |

## 📌 Summary
- **PEFT**: freeze the big model, train a tiny add-on
- **LoRA**: learn a low-rank update **B·A** beside each weight matrix — often < 1% of the parameters; saves as a small adapter you can switch on and off
- **QLoRA**: LoRA on top of a **4-bit** frozen model → fine-tune big models on small GPUs